In [8]:
import numpy as np
import pandas as pd
import time
import faiss
import hnswlib

from annoy import AnnoyIndex

In [9]:
def recall_at_k(I_pred, I_gt, k):
    correct = 0

    for i in range(len(I_pred)):
        correct += len(set(I_pred[i][:k]) & set(I_gt[i][:k]))

    return correct / (len(I_pred) * k)

In [10]:
def run_experiment(n_data, dim, nq=100, k=10, seed=42):
    rng = np.random.RandomState(seed)

    X = rng.random((n_data, dim)).astype('float32')
    Q = rng.random((nq, dim)).astype('float32')

    rows = []

    flat = faiss.IndexFlatL2(dim)
    flat.add(X)

    start = time.time()
    _, gt = flat.search(Q, k)
    t_query = (time.time() - start) * 1000 / nq

    rows.append({'Algorithm': 'Exact (FAISS Flat)', 'Build (s)': 0.0, 'Query (ms/query)': t_query, f'Recall@{k}': 1.0})

    ann = AnnoyIndex(dim, 'euclidean')

    start = time.time()
    for i in range(n_data):
        ann.add_item(i, X[i])
    ann.build(10)
    t_build = time.time() - start

    start = time.time()
    I = [ann.get_nns_by_vector(q, k) for q in Q]
    t_query = (time.time() - start) * 1000 / nq

    rows.append({'Algorithm': 'ANNOY', 'Build (s)': t_build, 'Query (ms/query)': t_query, f'Recall@{k}': recall_at_k(I, gt, k)})

    nlist = max(1, int(np.sqrt(n_data)))

    quantizer = faiss.IndexFlatL2(dim)
    ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)

    start = time.time()
    ivf.train(X)
    ivf.add(X)
    t_build = time.time() - start

    ivf.nprobe = 10

    start = time.time()
    _, I = ivf.search(Q, k)
    t_query = (time.time() - start) * 1000 / nq

    rows.append({'Algorithm': 'FAISS', 'Build (s)': t_build, 'Query (ms/query)': t_query, f'Recall@{k}': recall_at_k(I, gt, k)})

    hnsw = hnswlib.Index(space='l2', dim=dim)

    start = time.time()
    hnsw.init_index(max_elements=n_data, ef_construction=200, M=16)
    hnsw.add_items(X)
    t_build = time.time() - start

    hnsw.set_ef(50)

    start = time.time()
    I, _ = hnsw.knn_query(Q, k)
    t_query = (time.time() - start) * 1000 / nq

    rows.append({'Algorithm': 'HNSW', 'Build (s)': t_build, 'Query (ms/query)': t_query, f'Recall@{k}': recall_at_k(I, gt, k)})

    df = pd.DataFrame(rows)
    df.insert(0, 'Data/Dimensions', f'{n_data:,}/{dim}D')

    return df

In [11]:
n_big = 1_000_000

configs = [(1000, 2), (1000, 5), (n_big, 2), (n_big, 5)]

frames = []

for n_data, dim in configs:
    print(f'Running {n_data:,} data points, {dim}D ...')
    frames.append(run_experiment(n_data, dim))

results = pd.concat(frames, ignore_index=True)

results.round(4)

Running 1,000 data points, 2D ...
Running 1,000 data points, 5D ...
Running 1,000,000 data points, 2D ...
Running 1,000,000 data points, 5D ...


,Data/Dimensions,Algorithm,Build (s),Query (ms/query),Recall@10
0,"1,000/2D",Exact (FAISS Flat),0.0000,0.0894,1.000
1,"1,000/2D",ANNOY,0.0143,0.0229,0.993
2,"1,000/2D",FAISS,0.0016,0.0101,1.000
3,"1,000/2D",HNSW,0.0128,0.0139,1.000
4,"1,000/5D",Exact (FAISS Flat),0.0000,0.0000,1.000
5,"1,000/5D",ANNOY,0.0094,0.0297,0.962
6,"1,000/5D",FAISS,0.0000,0.0000,0.997
7,"1,000/5D",HNSW,0.0170,0.0100,1.000
8,"1,000,000/2D",Exact (FAISS Flat),0.0000,1.1704,1.000
9,"1,000,000/2D",ANNOY,9.0110,0.0399,0.998


In [12]:
query_table = results.pivot(index='Data/Dimensions', columns='Algorithm', values='Query (ms/query)')

query_table = query_table.loc[[f'{n:,}/{d}D' for n, d in configs]]

query_table.round(4)

Algorithm,ANNOY,Exact (FAISS Flat),FAISS,HNSW
Data/Dimensions,,,,
"1,000/2D",0.0229,0.0894,0.0101,0.0139
"1,000/5D",0.0297,0.0000,0.0000,0.0100
"1,000,000/2D",0.0399,1.1704,0.0470,0.0402
"1,000,000/5D",0.0500,1.9795,0.3124,0.0202


In [13]:
build_table = results.pivot(index='Data/Dimensions', columns='Algorithm', values='Build (s)')

build_table = build_table.loc[[f'{n:,}/{d}D' for n, d in configs]]

build_table.round(4)

Algorithm,ANNOY,Exact (FAISS Flat),FAISS,HNSW
Data/Dimensions,,,,
"1,000/2D",0.0143,0.0,0.0016,0.0128
"1,000/5D",0.0094,0.0,0.0000,0.0170
"1,000,000/2D",9.0110,0.0,10.3132,27.4544
"1,000,000/5D",8.3621,0.0,27.6922,49.2791


In [14]:
recall_table = results.pivot(index='Data/Dimensions', columns='Algorithm', values='Recall@10')

recall_table = recall_table.loc[[f'{n:,}/{d}D' for n, d in configs]]

recall_table.round(4)

Algorithm,ANNOY,Exact (FAISS Flat),FAISS,HNSW
Data/Dimensions,,,,
"1,000/2D",0.993,1.0,1.000,1.0
"1,000/5D",0.962,1.0,0.997,1.0
"1,000,000/2D",0.998,1.0,1.000,1.0
"1,000,000/5D",0.941,1.0,1.000,1.0


### Analysis – Lab Assignment 1

**Setup:** 100 random queries, k = 10, data uniform in [0, 1]. Exact search is FAISS `IndexFlatL2`; ANNOY uses 10 trees; FAISS uses `IndexIVFFlat` (`nlist` = sqrt(n), `nprobe` = 10); HNSW uses M = 16, `ef_construction` = 200, `ef` = 50. Times are measured with `time.time()`, which has a resolution of about 1 ms on this machine, so the per-query values for 1,000 points (0.00 to 0.03 ms) and ANNOY at 1,000,000/5D (0.0000) are at or below the timer resolution and should not be compared with each other.

**Answer (effect of the number of data points):** With 1,000 points, exact search already takes about 0.01 ms per query, so the approximate methods give no benefit. ANNOY is even slower (0.025 ms) and all three need a build step first (0.003 to 0.016 s). With 1,000,000 points, exact search grows to 1.05 ms (2D) and 1.27 ms (5D), while FAISS and HNSW answer in about 0.025 to 0.028 ms, which is roughly 40 to 50 times faster, and ANNOY in 2D needs 0.094 ms (about 11 times faster). The price is build time: about 7 to 8 s for ANNOY and FAISS and 20 to 33 s for HNSW, against practically none for exact search.

**Answer (effect of the dimension, 2D vs 5D):** The cost of exact search rises only slightly from 2D to 5D (1.05 ms to 1.27 ms). The recall of ANNOY drops clearly in 5D: from 0.993 to 0.962 with 1,000 points and from 0.998 to 0.941 with 1,000,000 points. FAISS stays at 1.000 for 1,000,000 points (0.997 for 1,000 points in 5D), and HNSW keeps a recall of 1.000 in all four settings. HNSW also needs more build time in 5D (19.7 s in 2D vs 32.7 s in 5D). A likely reason is that ANNOY with a fixed 10 trees and the default search budget misses more true neighbors in 5D, because the splitting hyperplanes separate close points more often, whereas the graph (HNSW) and the cluster-probing (FAISS) approaches are less affected.

**Answer (comparison):** HNSW is the most accurate (recall 1.000 in all settings) and has query times among the lowest, but it has the longest build time, so it fits best when the index is built once and queried many times. FAISS IVFFlat has almost the same query speed with a build time about 4 times shorter and a recall of at least 0.997, so it is the best balance in this experiment. ANNOY has the lowest recall in 5D at its default settings (0.94 to 0.96), and it would need more trees or a larger `search_k` to catch up, at the cost of build and query time. Exact search is the best choice for small data (1,000 points) because it is simple, has recall 1.0 and is just as fast; approximate search only pays off for large datasets or many queries.

**Note:** Using `time.perf_counter()` instead of `time.time()` would give meaningful per-query times for the small datasets. The conclusions above for 1,000,000 points do not depend on that.